In [ ]:
import subprocess, sys
required = ['shap', 'optuna', 'lightgbm', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']

for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg],
                       capture_output=True)
print("All packages ready.")

In [ ]:
# phase 1: data understanding
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

#load
df = pd.read_csv('application_train.csv')

print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")



In [ ]:
#basic inspection
print(df.dtypes.value_counts())
print()
print(df.head(3))


In [ ]:
#target variable analysis
target_counts = df['TARGET'].value_counts()
target_pct = df['TARGET'].value_counts(normalize=True)*100

print("Counts:")
print(target_counts)
print()
print("Percentages:")
print(target_pct.round(2))


In [ ]:
#phase 2: data cleaning

#missing value audit
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100 

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

#only show column that have missing values
missing_df = missing_df[missing_df['missing_count'] > 0]

print(f"Columns with missing values: {len(missing_df)} out of {df.shape[1]}")
print()
print(missing_df.head(20))



top missing columns are building features, people did not provide those info. 3 decisions that can be made for missing columns: 1. >50% drop column, 2. <10% impute (median for number, mode for category), 3. missing info is a feature, informative in itself. create _MISSING flag before imputing.

despite being >50%, EXT_SOURCES_1~3 are external credit scores from third party bureaus, which is typicall the strongest predictors of default in risk modelling. dropping them will significantly hurt model performance 

In [ ]:
#drop high-missing building/property columns
cols_to_drop = [col for col in df.columns if col in missing_df.index
                and missing_df.loc[col, 'missing_pct'] > 40
                and col not in ['EXT_SOURCE_1', 'OWN_CAR_AGE']]

df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns. New shape: {df.shape}")

#OWN_CAR_AGE — create missing flag, then fill
df['FLAG_OWN_CAR_AGE_MISSING'] = df['OWN_CAR_AGE'].isnull().astype(int)
df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].fillna(0)

#EXT_SOURCE - fill with median since they are important predictors 
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    df[col] = df[col].fillna(df[col].median())

#categorical - fill with 'Unknown'
df['NAME_TYPE_SUITE'] = df['NAME_TYPE_SUITE'].fillna('Unknown')
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

#almost no missing values — median impute
near_zero = ['AMT_GOODS_PRICE', 'AMT_ANNUITY', 'CNT_FAM_MEMBERS',
             'DAYS_LAST_PHONE_CHANGE', 'DEF_60_CNT_SOCIAL_CIRCLE',
             'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
             'OBS_30_CNT_SOCIAL_CIRCLE']

for col in near_zero:
    df[col] = df[col].fillna(df[col].median())

#AMT_REQ_CREDIT_BUREAU — fill with 0 since missing likely means no inquiries
bureau_cols = [col for col in df.columns if 'AMT_REQ_CREDIT_BUREAU' in col]
for col in bureau_cols:
    df[col] = df[col].fillna(0)

#final check
remaining_missing = df.isnull().sum().sum()
print(f"Total remaining missing values: {remaining_missing}")
print(f"Final shape: {df.shape}")

In [ ]:
#phase 3: EDA

#EXT_SOURCE distributions by target
fig, axes = plt.subplots(1, 3, figsize=(15,4))

for i, col in enumerate (['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']):
    axes[i].hist(df[df['TARGET'] == 0][col], bins=50, alpha=0.5, label='No Default', color='steelblue')
    axes[i].hist(df[df['TARGET'] == 1][col], bins=50, alpha=0.5, label='Default', color='tomato')
    axes[i].set_title(col)
    axes[i].legend()

plt.suptitle('EXT_SOURCE Distributions by Default Status', fontsize=13)
plt.tight_layout()
plt.show()

EXT_SOURCE_1 and EXT_SOURCE_3 — that giant spike at ~0.5 is the median we imputed. This is actually a problem we should note. A more sophisticated approach would use model-based imputation, but median is acceptable for now.

EXT_SOURCE_2 — this one is the most informative because it had very few missing values (0.2%), so the distribution is real. Notice that:

Non-defaulters (blue) peak at 0.6–0.8 — high credit scores
Defaulters (red) are flatter and shifted left — lower credit scores
This means higher EXT_SOURCE_2 → less likely to default

That separation between red and blue is what makes a feature predictive. The more separated the two distributions, the better the feature is for the model.

In [ ]:
#age vs default rate
#DAYS_BIRTH is negative (days before application), convert to age in years
df['AGE_YEARS'] = df['DAYS_BIRTH'] / -365

#bin into age groups
df['AGE_GROUP'] = pd.cut(df['AGE_YEARS'], bins=[20, 25, 30, 35, 40, 45, 50, 55, 60, 70])

#default rate per age group
age_default = df.groupby('AGE_GROUP', observed=True)['TARGET'].mean() * 100

#plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

#left — age distribution by default status
axes[0].hist(df[df['TARGET']==0]['AGE_YEARS'], bins=40, alpha=0.5, label='No Default', color='steelblue')
axes[0].hist(df[df['TARGET']==1]['AGE_YEARS'], bins=40, alpha=0.5, label='Default', color='tomato')
axes[0].set_title('Age Distribution by Default Status')
axes[0].set_xlabel('Age (Years)')
axes[0].legend()

#right — default rate by age group
axes[1].bar(age_default.index.astype(str), age_default.values, color='tomato', alpha=0.8)
axes[1].set_title('Default Rate (%) by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Default Rate (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(age_default.round(2))

DAYS_BIRTH in the raw dataset is a negative number - it records how many days before the application date the person was born. we flipped it with / -365 to get readable age in years

pd.cut() then splits the continuous age into buckets so we can compute default rate group (bivariate analysis, feature vs target)

In [ ]:
#income & loan amount vs default
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

#top left - income distribution by default status
axes[0,0].hist(df[df['TARGET']==0]['AMT_INCOME_TOTAL'].clip(upper=500000), 
               bins=50, alpha=0.5, label='No Default', color='steelblue')
axes[0,0].hist(df[df['TARGET']==1]['AMT_INCOME_TOTAL'].clip(upper=500000), 
               bins=50, alpha=0.5, label='Default', color='tomato')
axes[0,0].set_title('Income Distribution by Default Status')
axes[0,0].set_xlabel('Annual Income (clipped at 500k)')
axes[0,0].legend()

#top right - loan amount distribution by default status
axes[0,1].hist(df[df['TARGET']==0]['AMT_CREDIT'], 
               bins=50, alpha=0.5, label='No Default', color='steelblue')
axes[0,1].hist(df[df['TARGET']==1]['AMT_CREDIT'], 
               bins=50, alpha=0.5, label='Default', color='tomato')
axes[0,1].set_title('Loan Amount Distribution by Default Status')
axes[0,1].set_xlabel('Loan Amount (AMT_CREDIT)')
axes[0,1].legend()

#bottom left - credit to income ratio
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

axes[1,0].hist(df[df['TARGET']==0]['CREDIT_INCOME_RATIO'].clip(upper=10), 
               bins=50, alpha=0.5, label='No Default', color='steelblue')
axes[1,0].hist(df[df['TARGET']==1]['CREDIT_INCOME_RATIO'].clip(upper=10), 
               bins=50, alpha=0.5, label='Default', color='tomato')
axes[1,0].set_title('Credit-to-Income Ratio by Default Status')
axes[1,0].set_xlabel('Credit / Income (clipped at 10)')
axes[1,0].legend()

#bottom right - annuity to income ratio
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

axes[1,1].hist(df[df['TARGET']==0]['ANNUITY_INCOME_RATIO'].clip(upper=0.5), 
               bins=50, alpha=0.5, label='No Default', color='steelblue')
axes[1,1].hist(df[df['TARGET']==1]['ANNUITY_INCOME_RATIO'].clip(upper=0.5), 
               bins=50, alpha=0.5, label='Default', color='tomato')
axes[1,1].set_title('Annuity-to-Income Ratio by Default Status')
axes[1,1].set_xlabel('Annuity / Income (clipped at 0.5)')
axes[1,1].legend()

plt.suptitle('Income & Loan Features by Default Status', fontsize=13)
plt.tight_layout()
plt.show()

#summary stats
print(df.groupby('TARGET')[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 
                             'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO']].median().round(2))

.clip(upper=10) groups everyone with 10 or more ratio as 1 group. without clipping, outliers will be so far apart that meaningful part of distribution will be invisible. it makes the chart readable.

medians between defaulters and non-defaulters are close, income and loan amount are not seperated cleanyly. raw income isnt a strong predictor. 

CREDIT_INCOME_RATIO and ANNUITY_INCOME_RATIO were engineered here because absolute income means nothing without context. 5000 income is very different  if loan repayment is 500 vs 4500.

since raw AMT_INCOME_TOTAL shows minimal separation between classes, the engineered ratio features are more likely to carry predictive signal in the model

In [ ]:
#correlation heatmap
#select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

#compute correlation with TARGET, sort by absolute value
target_corr = df[numeric_cols].corr()['TARGET'].drop('TARGET')
target_corr_sorted = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)

#top 20 most correlated features with TARGET
top20 = target_corr_sorted.head(20)

#plot
fig, axes = plt.subplots(1, 2, figsize=(16,6))

#left - bar chart of top 20 correlations with TARGET
colors = ['tomato' if x < 0 else 'steelblue' for x in top20.values]
axes[0].barh(top20.index, top20.values, color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Top 20 Features Correlated with TARGET')
axes[0].set_xlabel('Pearson Correlation')
axes[0].invert_yaxis()

#right - heatmap of top 10 features vs each other
top10_cols = top20.head(10).index.tolist() + ['TARGET']
corr_matrix = df[top10_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], square=True)
axes[1].set_title('Correlation Matrix — Top 10 Features')

plt.tight_layout()
plt.show()

print("Top 10 correlations with TARGET:")
print(top20.head(10).round(4))



The strongest is EXT_SOURCE_2 at -0.16, this indicates they are not linearly separable problem. no 1 single feature cleanly predicts default. we will use lightGBM over simple logistic regression as it captures non-linear interactions between features. 

multicollinearity check - AGE_YEAR and DAYS_BIRTH show 1.00 correlation with each other, same feature. will drop DAYS_BIRTH before modelling to avoid duplication

REGION_RATING_CLIENT and REGION_RATING_CLIENT_W_CITY shows 0.95 correlation, almost identical. we will keep one and drop the other.



In [ ]:
#dropping redundant columns identified from correlation analysis
df = df.drop(columns=['DAYS_BIRTH', 'REGION_RATING_CLIENT'])
print(f"Shape after dropping redunddant columns: {df.shape}")

In [ ]:
#days_employed anomaly

print("DAYS_EMPLOYED basic stats:")
print(df['DAYS_EMPLOYED'].describe())
print()
print(f"Postive values in DAYS_EMPLOYED: {(df['DAYS_EMPLOYED']>0).sum()}")
print(f"Percentage: {(df['DAYS_EMPLOYED']>0).mean()*100:.2f}%")
print()

#plot distribution
fig, axes = plt.subplots(1,2,figsize=(14,4))

#left - full distribution
axes[0].hist(df['DAYS_EMPLOYED'], bins=50, color='steelblue', alpha=0.8)
axes[0].set_title('DAYS_EMPLOYED - Full Distribution')
axes[0].set_xlabel('Days Employed')

#right - exclusing the anomaly
axes[1].hist(df[df['DAYS_EMPLOYED']<0]['DAYS_EMPLOYED'],
                bins=50, color='steelblue', alpha=0.8)
axes[1].set_title('DAYS_EMPLOYED = Anomaly Removed')
axes[1].set_xlabel('Days Employed')

plt.tight_layout()
plt.show()

In [ ]:
anomaly_default = df[df['DAYS_EMPLOYED'] > 0]['TARGET'].mean() * 100
normal_default = df[df['DAYS_EMPLOYED'] < 0]['TARGET'].mean() * 100
print(f"Default rate — anomalous values: {anomaly_default:.2f}%")
print(f"Default rate — normal values: {normal_default:.2f}%")

In [ ]:
#flag the anomaly, then replace with NaN and impute
df['FLAG_EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median())

#convert to positive years for interpretability
df['YEARS_EMPLOYED'] = df['DAYS_EMPLOYED'] / -365

print(f"Anomaly flags created: {df['FLAG_EMPLOYED_ANOMALY'].sum()}")
print(f"Remaining nulls in DAYS_EMPLOYED: {df['DAYS_EMPLOYED'].isnull().sum()}")
print(f"YEARS_EMPLOYED sample stats:")
print(df['YEARS_EMPLOYED'].describe().round(2))

1. EXT_SOURCES 1/2/3 - strongest predictors, higher score = lower default
2. AGE_YEARS - younger = higher default
3. raw income/loan amount - by itself weak predictor
4. CREDIT_INCOME_RATIO - more predictive than raw values
5. DAYS_EMPLOYED anomaly - 365,243 = unemployed, not showing high risk
6. all correlation - weak (<0.20), non-linear model needed

In [ ]:
#phase 4: feature engineering

#credit burden features
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT']

#age & employment features
df['EMPLOYED_TO_AGE_RATIO'] = df['YEARS_EMPLOYED'] / df['AGE_YEARS']

#document submission rate
doc_cols = [col for col in df.columns if 'FLAG_DOCUMENT' in col]
df['DOCUMENT_SUBMISSION_RATE'] = df[doc_cols].sum(axis=1) / len(doc_cols)

#social circle risk
df['SOCIAL_CIRCLE_DEFAULT_RATE'] = (
    df['DEF_30_CNT_SOCIAL_CIRCLE'] + df['DEF_60_CNT_SOCIAL_CIRCLE']
) / (
    df['OBS_30_CNT_SOCIAL_CIRCLE'] + df['OBS_60_CNT_SOCIAL_CIRCLE'] + 1
)

#credit bureau enquiry tool
bureau_cols = [col for col in df.columns if 'AMT_REQ_CREDIT_BUREAU' in col]
df['TOTAL_CREDIT_ENQUIRIES'] = df[bureau_cols].sum(axis=1)

#verify
new_features = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
                'GOODS_CREDIT_RATIO', 'EMPLOYED_TO_AGE_RATIO', 
                'DOCUMENT_SUBMISSION_RATE', 'SOCIAL_CIRCLE_DEFAULT_RATE',
                'TOTAL_CREDIT_ENQUIRIES']

print(f"New features created: {len(new_features)}")
print()
print(df[new_features].describe().round(3))

In [ ]:
#phase 5: pre-processing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

#step 1:check remaining categorical columns
cat_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Categorical columns: {len(cat_cols)}")
print(cat_cols)

In [ ]:
#check unique values per categorical column
for col in cat_cols:
    print(f"{col:40s} {df[col].nunique()} unique values")

In [ ]:
#step 2: fix CODE_GENDER XNA
print("CODE_GENDER value counts:")
print(df['CODE_GENDER'].value_counts())

#replace XNA with mode
df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', df['CODE_GENDER'].mode()[0])

#step 3: label encode all categorical columns
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(f"\nEncoding done. Sample:")
print(df[cat_cols].head(3))

#step 4: drop columns we no longer need
# AGE_GROUP was for EDA only, not for modelling
# DAYS_EMPLOYED replaced by YEARS_EMPLOYED
df = df.drop(columns=['AGE_GROUP', 'DAYS_EMPLOYED'], errors='ignore')

print(f"\nShape after cleanup: {df.shape}")


In [ ]:
#step 5: separate features and target
x = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print(f"Features shape: {x.shape}")
print(f"Target shape: {y.shape}")

#step 6: stratified train/test split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {x_train.shape}")
print(f"Test size: {x_test.shape}")
print(f"\nTarget distribution in train:")
print(y_train.value_counts(normalize=True).round(4) * 100)
print(f"\nTarget distribution in test:")
print(y_test.value_counts(normalize=True).round(4) * 100)

In [ ]:
#step 7: Calculate scale_pos_weight for LightGBM
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Non-defaulters (train): {neg}")
print(f"Defaulters (train): {pos}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:

#phase 6: modelling (lightGBM)

import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, RocCurveDisplay
)
import warnings
warnings.filterwarnings('ignore')

#step 1: baseline lightGBM 
lgb_params = {
    'objective':        'binary',
    'metric':           'auc',
    'boosting_type':    'gbdt',
    'n_estimators':     1000,
    'learning_rate':    0.05,
    'num_leaves':       31,
    'max_depth':        -1,
    'min_child_samples': 20,
    'scale_pos_weight': scale_pos_weight,   # class imbalance fix
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'random_state':     42,
    'n_jobs':           -1,
    'verbose':          -1,
}

model = lgb.LGBMClassifier(**lgb_params)

model.fit(
    x_train, y_train,
    eval_set=[(x_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

print(f"\nBest iteration: {model.best_iteration_}")

In [ ]:
#step 2: evaluate
y_pred_proba = model.predict_proba(x_test)[:, 1]
y_pred = model.predict(x_test)

auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC-AUC Score: {auc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))


In [ ]:
#step 3: visualize - ROC curve + confusion matrix
fig, axes = plt.subplots(1,2, figsize=(14,5))

#left - ROC curve
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], color='steelblue')
axes[0].plot([0,1], [0,1], 'k--', lw=1)
axes[0].set_title(f'ROC Curve (AUC = {auc:.4f})')

#right - confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Pred: No Default', 'Pred: Default'],
            yticklabels=['True: No Default', 'True: Default'])
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
#step 4: feature importance
importance_df = pd.DataFrame({
    'feature': x_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

#top 25
top25 = importance_df.head(25)

fig, ax = plt.subplots(figsize=(10,8))
ax.barh(top25['feature'], top25['importance'], color='steelblue', alpha=0.8)
ax.invert_yaxis()
ax.set_title('Top 25 Feature Importances (LightGBM gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print(top25.to_string(index=False))




In [ ]:
#step 5: threshold tuning (precision-recall tradeoff)
from sklearn.metrics import precision_recall_curve, f1_score

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Default threshold (0.5) F1: {f1_score(y_test, (y_pred_proba >= 0.5).astype(int)):.4f}")
print(f"Optimal threshold: {best_threshold:.4f}")
print(f"Optimal threshold F1: {best_f1:.4f}")

fig, ax = plt.subplots(figsize=(10,4))

ax.plot(thresholds, f1_scores[:-1], color='steelblue')
ax.axvline(best_threshold, color='tomato', linestyle='--',
           label=f'Best threshold = {best_threshold:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score vs Decision Threshold')
ax.legend()
plt.tight_layout()
plt.show()           



In [ ]:
#phase 7(A): bureau table aggregation

bureau = pd.read_csv('bureau.csv')
bureau_balance = pd.read_csv('bureau_balance.csv')

print(f"Bureau shape: {bureau.shape}")
print(f"Bureau balance shape: {bureau_balance.shape}")
print()
print(bureau.head(3))



In [ ]:
#aggregate bureau_balance up to bureau level
bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(
    MONTHS_COUNT = ('MONTHS_BALANCE','count'),
    MONTHS_MIN = ('MONTHS_BALANCE', 'min'),
    STATUS_C_COUNT = ('STATUS', lambda x: (x == 'C').sum()), #closed accounts count
    STATUS_X_COUNT = ('STATUS', lambda x: (x == 'X').sum()), #no info count
    STATUS_0_COUNT = ('STATUS', lambda x: (x == '0').sum()),#no DPD count
    STATUS_1_COUNT = ('STATUS', lambda x: (x == '1').sum()), #1-30 DPD count
    STATUS_2_PLUS_COUNT = ('STATUS', lambda x: x.isin(['2','3','4','5']).sum()), #30+ DPD count
).reset_index()

#DPD rate - how often was this loan delinquent
bb_agg['DPD_RATE'] = bb_agg['STATUS_2_PLUS_COUNT']/(bb_agg['MONTHS_COUNT']+1)

print(f"Bureau balance aggregated shape: {bb_agg.shape}")
print(bb_agg.head(3))

In [ ]:
#merge bureau_balance agg into bureau
bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

#aggregate bureau up to application level
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT = ('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT = ('CREDIT_ACTIVE', lambda x: (x=='Active').sum()),
    BUREAU_CLOSED_COUNT = ('CREDIT_ACTIVE', lambda x: (x=='Closed').sum()),

    #amounts
    BUREAU_DEBT_SUM = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_DEBT_MEAN = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    BUREAU_CREDIT_SUM = ('AMT_CREDIT_SUM', 'sum'),
    BUREAU_CREDIT_MEAN = ('AMT_CREDIT_SUM', 'mean'),
    BUREAU_OVERDUE_SUM = ('AMT_CREDIT_SUM_OVERDUE','sum'),

    #recency
    BUREAU_DAYS_CREDIT_MEAN = ('DAYS_CREDIT','mean'),
    BUREAU_DAYS_CREDIT_MIN = ('DAYS_CREDIT','min'),
    BUREAU_DAYS_ENDDATE_MEAN = ('DAYS_CREDIT_ENDDATE','mean'),
    
    #deliquency from bureau_balnce
    BUREAU_DPD_RATE_MEAN = ('DPD_RATE','mean'),
    BUREAU_DPD_RATE_MAX = ('DPD_RATE','max'),
    BUREAU_STATUS_2PLUS_SUM = ('STATUS_2_PLUS_COUNT', 'sum'),                     
).reset_index()

#derived ratios
bureau_agg['BUREAU_ACTIVE_RATIO'] = bureau_agg['BUREAU_ACTIVE_COUNT']/(bureau_agg['BUREAU_LOAN_COUNT']+1)
bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = bureau_agg['BUREAU_DEBT_SUM']/(bureau_agg['BUREAU_CREDIT_SUM']+1)

print(f"Bureau aggregated shape: {bureau_agg.shape}")
print(bureau_agg.describe().round(2))

In [ ]:
#merge into main df
#reload df splits - we need to re-merge before splitting again 
#so we merge at the full df level first
 
#if df is still in memory:
df_enriched = df.merge(bureau_agg, on='SK_ID_CURR', how='left')

#full null for applicants with no bureau records (new-to-credit)
bureau_cols_new = [c for c in bureau_agg.columns if c != 'SK_ID_CURR']
df_enriched[bureau_cols_new] = df_enriched[bureau_cols_new].fillna(0)

print(f"Enriched shape: {df_enriched.shape}")
print(f"New bureau features added: {len(bureau_cols_new)}")


In [ ]:
#re-split with enriched features
x2 = df_enriched.drop(columns=['TARGET', 'SK_ID_CURR'])
y2 = df_enriched['TARGET']

x2_train, x2_test, y2_train, y2_test = train_test_split(
    x2, y2, test_size=0.2, random_state=42, stratify=y2
)

print(f"Enriched train shape: {x2_train.shape}")

In [ ]:
#retrain and compare RUC
model2 = lgb.LGBMClassifier(**lgb_params)

model2.fit(
    x2_train, y2_train,
    eval_set = [(x2_test, y2_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

y2_pred_proba = model2.predict_proba(x2_test)[:,1]
auc2 = roc_auc_score(y2_test, y2_pred_proba)

print(f"\nBaseline AUC (application only): {auc:.4f}")
print(f"Enriched AUC (+ bureau features): {auc2:.4f}")
print(f"Delta: +{auc2-auc:.4f}")

In [ ]:
#phase 7B: SHAP explainability

import shap

#use a sample for speed - SHAP on full test set is slow
sample_idx = x2_test.sample(n=2000, random_state=42).index
x_shap = x2_test.loc[sample_idx]

explainer = shap.TreeExplainer(model2)
shap_values = explainer.shap_values(x_shap)

print("SHAP values computed.")
print(f"Shape: {np.array(shap_values).shape}")


In [ ]:
#summary plot - global feature importance
plt.figure()
shap.summary_plot(shap_values, x_shap, max_display=20, show=False)
plt.title('SHAP Summary - Top 20 Features')
plt.tight_layout()
plt.show()

In [ ]:
#bar_plot - mean absolute SHAP
plt.figure()
shap.summary_plot(shap_values, x_shap, plot_type='bar',
                  max_display=20, show=False)
plt.title('SHAP Mean Absolute Importance')
plt.tight_layout()
plt.show()


In [ ]:
#dependence plot - EXT_SOURCE_2 (strongest predictor)

plt.figure()
shap.dependence_plot('EXT_SOURCE_2', shap_values, x_shap, interaction_index='EXT_SOURCE_3', show=False)

plt.title('SHAP Dependence - EXT_SOURCE_2 vs EXT_SOURCE_3')
plt.tight_layout()
plt.show()

In [ ]:
#waterfall - single prediction explanation

defaulter_idx = y2_test.loc[sample_idx][y2_test.loc[sample_idx] == 1].index[0]

pos=list(x_shap.index).index(defaulter_idx)

explanation = shap.Explanation(
    values= shap_values[pos],
    base_values = explainer.expected_value,
    data = x_shap.loc[defaulter_idx].values,
    feature_names = x_shap.columns.tolist()
)

shap.plots.waterfall(explanation, max_display=15, show=True)



In [ ]:
print(f"Baseline AUC:  {auc:.4f}")
print(f"Enriched AUC:  {auc2:.4f}")
print(f"model2 best iteration: {model2.best_iteration_}")
print(f"x2_train shape: {x2_train.shape}")
print(f"Remaining nulls: {df_enriched.isnull().sum().sum()}")

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"Optuna version: {optuna.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Training set size: {x2_train.shape}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42,
        'scale_pos_weight': scale_pos_weight,

        'n_estimators': trial.suggest_int('n_estimators',300,1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }

    model_cv = lgb.LGBMClassifier(**params)
    model_cv.fit(
        x2_train, y2_train,
        eval_set = [(x2_test, y2_test)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )

    preds = model_cv.predict_proba(x2_test)[:,1]
    return roc_auc_score(y2_test, preds)

print("Objective function defined.")

In [ ]:
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
best_params = study.best_params
best_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42,
    'scale_pos_weight': scale_pos_weight,
})

model_final = lgb.LGBMClassifier(**best_params)
model_final.fit(
    x2_train, y2_train,
    eval_set=[(x2_test, y2_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

y_final_proba = model_final.predict_proba(x2_test)[:,1]
auc_final = roc_auc_score(y2_test, y_final_proba)

print(f"\nBaseline AUC (app only, default params): {auc:.4f}")
print(f"Enriched AUC (+ bueau, default params): {auc2:.4f}")
print(f"Tuned AUC (+bureau, tuned params): {auc_final:.4f}")
print(f"\nTotal gain from Phase baseline: +{auc_final - auc:.4f}")

## Phase 7 Summary

### AUC Progression
| Stage | AUC |
|---|---|
| Baseline — application_train.csv only | 0.7706 |
| + Bureau feature aggregation | 0.7740 |
| + Optuna hyperparameter tuning (50 trials) | 0.7766 |

### Key Findings
- Bureau features contributed +0.0034 lift — BUREAU_DEBT_CREDIT_RATIO and BUREAU_DPD_RATE confirmed as meaningful signals via SHAP
- Optimal model uses 936 trees at learning_rate=0.017 with num_leaves=74 — slower and deeper than the default configuration
- SHAP waterfall confirmed EXT_SOURCE_2 and EXT_SOURCE_3 remain the dominant predictors at the individual prediction level
- Total gain of +0.0059 AUC from a single additional table and tuning — further tables (installments, previous applications) expected to yield additional lift

### Limitations
- bureau.csv only — 5 additional tables unused
- Single train/test split — no cross-validation yet
- Median imputation on EXT_SOURCE_1/3 introduces spike artifact